# **1. ZAPOZNANIE ZE ZBIOREM**
- Wczytaj zbiór danych
- Zlicz ile rekordów znajduje się w zbiorze danych.
- Ile różnych gier znajduje się w zbiorze danych i ile razy każda z tych gier się pojawia  
- Zlicz ile jest unikalnych tagów oraz ile tagów jest przypisanych tylko do jednej gry, a ile do kilku
- Wyświetl 20 najczęściej występujących tagów wraz z liczbą wystąpień


In [13]:
import json
from collections import Counter

# 1.Wczytaj zbiór danych
with open("tagi_gier_wspoldzielone.json", encoding="utf-8") as f:
  data = json.load(f)


# 2 Zlicz ile rekordów znajduje się w zbiorze danych.
#   Ile różnych gier znajduje się w zbiorze danych i ile razy każda z tych gier się pojawia
game_count = Counter()

for record in data:
    label = record["label"]
    game_count[label] += 1

print("Liczba rekordów:", len(data))
print("Liczba unikalnych gier:", len(game_count))

print("\nLiczba rekordów dla każdej gry:")

for label, count in game_count.items():
    print(" -", label, ":", count)


# 3. Zlicz ile jest unikalnych tagów oraz ile tagów jest przypisanych tylko do jednej gry, a ile do kilku
tag_count = Counter()
tag_to_games = {}

for record in data:
    label = record["label"]
    for tag in record["tags"]:
        tag_count[tag] += 1

        if tag not in tag_to_games:
            tag_to_games[tag] = set()

        tag_to_games[tag].add(label)


# Zliczenie tagów przypisanych do jednej lub wielu gier
tags_for_one_game = 0
tags_for_many_games = 0

for tag, games in tag_to_games.items():
    if len(games) == 1:
        tags_for_one_game += 1
    else:
        tags_for_many_games += 1

print("\nLiczba unikalnych tagów:", len(tag_count))
print("Tagi przypisane tylko do jednej gry:", tags_for_one_game)
print("Tagi wspólne dla kilku gier:", tags_for_many_games)


# 4. Wyświetl 20 najczęściej występujących tagów wraz z liczbą wystąpień
print("\n20 najczęściej występujących tagów:")
for tag, count in tag_count.most_common(20):
    print(" -", tag, ":", count, "razy")

Liczba rekordów: 300
Liczba unikalnych gier: 5

Liczba rekordów dla każdej gry:
 - minecraft : 60
 - fortnite : 60
 - roblox : 60
 - lol : 60
 - cs : 60

Liczba unikalnych tagów: 74
Tagi przypisane tylko do jednej gry: 25
Tagi wspólne dla kilku gier: 49

20 najczęściej występujących tagów:
 - roleplay : 50 razy
 - crafting : 49 razy
 - budowanie : 48 razy
 - klocki : 46 razy
 - sezon : 46 razy
 - emotki : 46 razy
 - loot : 44 razy
 - rankedy : 44 razy
 - kopanie : 43 razy
 - oddziały : 43 razy
 - karabin : 41 razy
 - arena : 41 razy
 - pistolet : 40 razy
 - waluta : 40 razy
 - questy : 39 razy
 - fortyfikacje : 39 razy
 - symulator : 39 razy
 - aktualizacja : 39 razy
 - kolekcje : 38 razy
 - tryb solo : 38 razy


# **2. PRZYGOTOWANIE MODELU**

## Czego musi nauczyć się model?

Po podziale danych model będzie uczony wyłącznie na zbiorze treningowym.

Podczas treningu model musi wyznaczyć:

1. listę możliwych klas,
2. liczbę wystąpień każdej klasy,
3. prawdopodobieństwo początkowe każdej klasy,
4. słownik tagów występujących w treningu,
5. liczbę wystąpień każdego tagu w każdej klasie,
6. łączną liczbę tagów w każdej klasie.

Na podstawie tych informacji model będzie obliczał, która gra
jest najbardziej prawdopodobna dla nowego zestawu tagów.

Zbiór testowy nie uczestniczy w uczeniu. Służy wyłącznie do
sprawdzenia, czy model poprawnie klasyfikuje nowe rekordy.

In [14]:
import json
import random
import math
from sklearn.model_selection import train_test_split

# Wczytanie pliku JSON do listy rekordów
def load_data(path):
    """Zwraca listę słowników w postaci {label: ..., tags: [...]}."""
    with open(path, encoding="utf-8") as file:
        return json.load(file)


# Podział na zbiór treningowy i testowy (80 / 20)
def split_data(data, test_ratio=0.20, random_state=42):
    labels = []

    for record in data:
        labels.append(record["label"])

    train, test = train_test_split(
        data,
        test_size=test_ratio,
        random_state=random_state,
        stratify=labels
    )
    print(f"Liczba próbek zbioru treningowego: {len(train)}")
    print(f"Liczba próbek zbioru testowego: {len(test)}")
    return train, test


# Budowa słownika wszystkich tagów występujących w treningu
def build_vocabulary(train):
    vocabulary = set()

    for record in train:
        vocabulary.update(record["tags"])

    print(f"Liczba tagów znanych modelowi: {len(vocabulary)}")
    return vocabulary


# Trenowanie modelu klasyfikatora Naive Bayes
def train_nb(train, vocab, alpha=1.0):
    class_counts = {}# Liczba rekordów każdej klasy
    tag_counts = {}# Liczba wystąpień każdego tagu w każdej klasie
    total_tags = {}# Łączna liczba tagów w każdej klasie

    for record in train:
        class_name = record["label"]

        # Zliczenie rekordów należących do danej klasy
        class_counts[class_name] = (class_counts.get(class_name, 0) + 1)

        # Utworzenie pustych struktur dla nowej klasy
        tag_counts.setdefault(class_name, {})
        total_tags.setdefault(class_name, 0)

        # Zliczenie tagów występujących w danej klasie
        for tag in record["tags"]:
            tag_counts[class_name][tag] = (tag_counts[class_name].get(tag, 0) + 1)
            total_tags[class_name] += 1


    # Zapisanie informacji zdobytych podczas treningu
    model = {
        "class_counts": class_counts,
        "tag_counts": tag_counts,
        "total_tags": total_tags,
        "vocab": vocab,
        "alpha": alpha,
        "total_records": len(train)
    }

    print("Model został wytrenowany.")
    print("Liczba klas:", len(class_counts))
    print("Liczba rekordów treningowych:", len(train))
    print("Liczba znanych tagów:", len(vocab))

    return model


# Obliczanie log-prawdopodobieństwa jednej klasy dla rekordu
def log_prob(model, record, class_name):
    # Prawdopodobieństwo początkowe klasy P(C)
    class_probability = (model["class_counts"][class_name]/ model["total_records"])
    log_probability = math.log(class_probability)

    # Liczba wszystkich tagów znanych modelowi
    vocabulary_size = len(model["vocab"])

    # Parametr wygładzania Laplace'a
    alpha = model["alpha"]

    # Obliczanie prawdopodobieństwa kolejnych tagów
    for tag in record["tags"]:

        # Pominięcie tagów, których nie było w zbiorze treningowym
        if tag not in model["vocab"]:
            continue

        # Liczba wystąpień tagu w analizowanej klasie
        tag_count = model["tag_counts"][class_name].get(tag, 0)

        # Prawdopodobieństwo P(tag | klasa)
        tag_probability = ((tag_count + alpha)/(model["total_tags"][class_name] + alpha * vocabulary_size))
        log_probability += math.log(tag_probability)

    return log_probability


# Predykcja – wybieramy klasę z najwyższym log-prawdopodobieństwem
def predict(model, record):
    best_class = None
    best_log_probability = -math.inf

    # Obliczenie wyniku dla każdej możliwej klasy
    for class_name in model["class_counts"]:

        current_log_probability = log_prob(
            model,
            record,
            class_name
        )

        if current_log_probability > best_log_probability:
            best_class = class_name
            best_log_probability = current_log_probability

    return best_class


# Ewaluacja modelu na zbiorze testowym
def evaluate(model, test):
    correct = 0

    for record in test:
        predicted_class = predict(model, record)
        real_class = record["label"]

        if predicted_class == real_class:
            correct += 1

    accuracy = correct / len(test)

    print(f"Poprawne przewidywania: {correct}/{len(test)}")
    print(f"Dokładność (accuracy): {accuracy:.2%}")
    return accuracy

In [15]:
# Główna funkcja programu
def main():
    path = "tagi_gier_wspoldzielone.json"
    data = load_data(path)# Wczytanie całego zbioru danych
    train, test = split_data(data, test_ratio=0.20)# Podział danych na zbiór treningowy i testowy
    vocab = build_vocabulary(train)# Budowa słownika tagów wyłącznie na podstawie zbioru treningowego
    model = train_nb(train, vocab, alpha=1.0)# Trenowanie modelu

    sample = test[0] # Wybranie przykładowego rekordu ze zbioru testowego

    # Wykonanie pojedynczej predykcji
    predicted_class = predict(model, sample)

    print("\nPrzykładowa predykcja:")
    print("Tagi:", sample["tags"])
    print("Rzeczywista gra:", sample["label"])
    print("Model przewidział:", predicted_class)

    # Ocena modelu na całym zbiorze testowym
    print("\nEwaluacja modelu:")
    evaluate(model, test)

main()

Liczba próbek zbioru treningowego: 240
Liczba próbek zbioru testowego: 60
Liczba tagów znanych modelowi: 74
Model został wytrenowany.
Liczba klas: 5
Liczba rekordów treningowych: 240
Liczba znanych tagów: 74

Przykładowa predykcja:
Tagi: ['aktualizacja', 'esport', 'tryb solo', 'kopanie', 'robuxy', 'lua', 'czat', 'parkour']
Rzeczywista gra: roblox
Model przewidział: roblox

Ewaluacja modelu:
Poprawne przewidywania: 60/60
Dokładność (accuracy): 100.00%
